### Our dataset need to have these column in order: 
Year
Title
Origin
Director
Cast
Genre
Plot

In [4]:
import pandas as pd
import re

### Input your input and output file's name here

In [5]:
# Đường dẫn trỏ vào thư mục raw chứa dữ liệu gốc
input_file = '../data/raw/t1_raw.csv' 


output_file = '../data/processed/t1.csv' 

df = pd.read_csv(input_file)

In [3]:
df.columns


Index(['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast',
       'Genre', 'Wiki Page', 'Plot'],
      dtype='str')

### If you want to delete Column (by their name), run code below

In [ ]:
if 'Column name' in df.columns:
    df = df.drop(columns=['Column name'])
    print("Deleted!")
else:
    print("No Column Found!")

### If you want to change Column's name, run code below

In [ ]:
if 'Column old name' in df.columns:
    df = df.rename(columns={'Column old name': 'Column new name'})
    print("Name changed!")
else:
    print("No Column Found!")

### If you want to delete duplicated Row or Film that appeared more than once, run code below

In [3]:
original_count = len(df)
df = df.drop_duplicates(subset=['Title'], keep='first')
removed_count = original_count - len(df)
if removed_count > 0:
    print("Deleted!")

### Instead of delete, you can merge two rows of same Title

In [ ]:
def count_words(text):
    if pd.isna(text) or text == '':
        return 0
    return len(str(text).split())

grouped = df.groupby('Title', as_index=False).agg({
    'Year': 'first',
    'Origin': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'Director': 'first',
    'Cast': 'first',
    'Genre': 'first',
    'Plot': lambda x: ' '.join(x.astype(str))  # Merge tất cả plot
})

def merge_values(group):
    result = {}
    for col in ['Year', 'Origin', 'Director', 'Cast', 'Genre']:
        values = group[col].tolist()
        non_unknown = [v for v in values if v != 'unknown']
        if len(non_unknown) > 0:
            result[col] = max(non_unknown, key=count_words)
        else:
            result[col] = 'unknown'

    plots = group['Plot'].tolist()
    result['Plot'] = ' '.join([p for p in plots if p != ''])
    
    return pd.Series(result)

df_merged = df.groupby('Title', as_index=False).apply(merge_values, include_groups=False).reset_index(drop=True)

print("Merged!")

### If there is not plot summarize, delete that row or you can add summarize yourself. If you want to spot and delete, run code below

In [4]:
if 'Plot' in df.columns:
    before_drop = len(df)
    df = df[df['Plot'].notna()] 
    df = df[df['Plot'].astype(str).str.strip() != '']
    removed_plot = before_drop - len(df)
    if removed_plot > 0:
        print("Deleted!")
else:
    print("No Plot Column Found!")

### Lowercase data

In [5]:
for col in df.columns:
    df[col] = df[col].apply(lambda x: x.lower() if pd.notna(x) and isinstance(x, str) else x)
print("Lowercase done!")

Lowercase done!


### Delete Special Charatecs

In [8]:
special_chars = r'[!@#$%&^*()\'"{}[\$\\|+_=`~/<>?-]'
for col in df.columns:
    df[col] = df[col].apply(
        lambda x: re.sub(special_chars, '', str(x)) if pd.notna(x) else x
    )
print("Deleted!")

Deleted!


### Delete unneccesary space

In [9]:
for col in df.columns:
    df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip() if pd.notna(x) else x)
print("Space Deleted!")

Space Deleted!


### If you want delete rows that don't have enough word, run code below

In [10]:
df['word_count'] = df['Plot'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
before = len(df)
df = df[df['word_count'] >= 2]
df = df.drop(columns=['word_count'])
print("Deleted!")

Deleted!


### There will be some information missing or unknown (except column Plot), you can fill in the cell unknown. Run code below to fill unknown to NaN cell

In [11]:
for col in df.columns:
    if col == 'Plot':
        df[col] = df[col].fillna('')
    else:
        df[col] = df[col].fillna('unknown')
        df[col] = df[col].replace('', 'unknown')
print("Filled!")

Filled!


### Sort Column

In [ ]:
column_order = ['Year', 'Title', 'Origin', 'Director', 'Cast', 'Genre', 'Plot']
df = df[column_order]
print("Sorted!")

### To export file, run code below

In [6]:
df.to_csv(output_file, index=False, encoding='utf-8-sig')